# Preprocess: clean Watkins → splits → spectrograms

Runs the whole data pipeline. Each step is a function in `src/preprocess/`, so it can also run from the command line.

1. **Clean** (`clean.py`): `old/index.csv` + raw wavs → `Annotations/audio_annotations.csv` + `Annotations/excluded.csv`
2. **Split** (`split.py`): main split (44 species, clip-level) and grouped split (34 species, by recording)
3. **Spectrograms** (`preprocess.py`): one `.norm.pt` per clean clip in `Spectrograms/<category>/`

The raw audio is never modified.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import config
print("data root:", config.DATA_ROOT)

data root: D:\UNI&WORK\Watkins


## 1. Clean annotations

Dropped (first matching reason wins): `no_audio`, `species_not_in_44`, `multi_species` (2 species audible, 2025 stored it under both),
`label_conflict` (2025 label ≠ download label, mostly White-beaked vs White-sided), `unreadable`, `too_short` (< 0.2 s),
`duplicate_audio` / `duplicate_conflicting` (byte-identical files).
Flagged but kept: `sr_flag` = `low` (< 4 kHz) / `high` (> 96 kHz).

In [2]:
from src.preprocess.clean import build_clean_annotations
clean, excluded = build_clean_annotations()
clean.head()

reading audio:   0%|          | 0/14383 [00:00<?, ?it/s]

reading audio:  17%|█▋        | 2403/14383 [00:30<02:29, 80.06it/s]

reading audio:  17%|█▋        | 2403/14383 [00:40<02:29, 80.06it/s]

reading audio:  39%|███▉      | 5659/14383 [01:00<01:30, 96.78it/s]

reading audio:  39%|███▉      | 5659/14383 [01:10<01:30, 96.78it/s]

reading audio:  57%|█████▋    | 8189/14383 [01:30<01:08, 91.05it/s]

... (7 outputs left out)

reading audio:  91%|█████████ | 13089/14383 [03:30<00:28, 45.11it/s]

reading audio: 100%|██████████| 14383/14383 [03:46<00:00, 63.42it/s]

clean clips: 14098 | species: 44 | recordings: 378
excluded by reason:
reason
label_conflict       535
multi_species        299
too_short            272
species_not_in_44     31
duplicate_audio       13
no_audio               1
sample-rate flags: {'ok': 11554, 'high': 1874, 'low': 670}


,category,filename,path,recording,year,duration_sec,sample_rate,channels,sha1,sr_flag
0,AtlanticSpottedDolphin,61025001.wav,AtlanticSpottedDolphin/1961/61025001.wav,61025,1961.0,3.527995,43900,1,78246da28ccbbac0fd705bf5d551a52f62fa2321,ok
1,AtlanticSpottedDolphin,61025002.wav,AtlanticSpottedDolphin/1961/61025002.wav,61025,1961.0,3.527995,43900,1,235b527704b5de9555a6045196ca8ef09c9c33d4,ok
2,AtlanticSpottedDolphin,61025003.wav,AtlanticSpottedDolphin/1961/61025003.wav,61025,1961.0,3.527995,43900,1,1208892be50472001e9dcc9c840373e1bc5307d8,ok
3,AtlanticSpottedDolphin,61025004.wav,AtlanticSpottedDolphin/1961/61025004.wav,61025,1961.0,1.763986,43900,1,ac3f76f45a5f1b79a9268e8a585834177045493f,ok
4,AtlanticSpottedDolphin,61025005.wav,AtlanticSpottedDolphin/1961/61025005.wav,61025,1961.0,1.323007,43900,1,63dfb5d3b05dcaeddcbf019912f6376dacfb7e64,ok


In [3]:
excluded.groupby("reason").agg(clips=("filename", "size"), species=("zip_species", lambda s: ", ".join(sorted(set(map(str, s))))[:120]))

,clips,species
reason,,
duplicate_audio,13,"Fin_FinbackWhale, KillerWhale, NorthernRightWh..."
label_conflict,535,"HarbourSeal, SpermWhale, White_sidedDolphin"
multi_species,299,"BowheadWhale, RibbonSeal, RingedSeal, RossSeal..."
no_audio,1,nan
species_not_in_44,31,"Commerson'sDolphin, FinlessPorpoise, GraySeal,..."
too_short,272,"Beluga_WhiteWhale, CommonDolphin, Dall'sPorpoi..."


## 2. Splits

* **Main split: 44 species, clip-level**, stratified 70/15/15, seed 42 (same method as 2025, on clean data).
  Clips of one recording can be in train *and* test → measures recognising species in partly *seen* recordings.
* **Grouped split: 34 species, by recording.** A recording (first 5 characters of the filename = Watkins tape/session)
  is entirely in one set → measures recognising species in *unseen* recordings (Mohammed's generalization question).
  Species with < 3 recordings cannot be in all three sets and are left out (`grouped_34/species_left_out.csv`).

In [4]:
from src.preprocess.split import make_all_splits
main, grouped, dropped = make_all_splits()

main (44 species, clip-level): [9868, 2115, 2115]


grouped (34 species, by recording): [9502, 2139, 2020] shares: [0.696, 0.157, 0.148]
left out of grouped split: {'Boutu_AmazonRiverDolphin': 2, 'Dall_sPorpoise': 1, 'DuskyDolphin': 1, 'Heaviside_sDolphin': 1, 'LeopardSeal': 2, 'LongBeaked(Pacific)CommonDolphin': 2, 'MelonHeadedWhale': 2, 'Rough_ToothedDolphin': 2, 'SouthernRightWhale': 2, 'TucuxiDolphin': 1}


In [5]:
import pandas as pd
pd.read_csv(config.GROUPED_SPLIT_DIR / "split_summary.csv", index_col=0)

,train_clips,train_recordings,val_clips,val_recordings,test_clips,test_recordings
category,,,,,,
AtlanticSpottedDolphin,183,1,54,1,7,1
BeardedSeal,43,3,10,2,10,2
Beluga_WhiteWhale,103,8,22,2,22,2
BottlenoseDolphin,128,3,29,2,26,1
BowheadWhale,210,2,18,2,115,1
ClymeneDolphin,173,1,153,1,2,1
CommonDolphin,614,5,129,2,130,2
FalseKillerWhale,370,2,70,1,64,1
Fin_FinbackWhale,400,28,84,6,86,7


## 3. Spectrograms

44 kHz mono, first ~5.2 s, mel spectrogram (n_fft 1024, hop 512, 128 mels), power → dB, per-clip z-score → `[1, 128, T]` float32.
The dataset class pads/crops to 400 frames (~4.65 s), as in 2025.

In [6]:
from src.preprocess.preprocess import generate_all
errors = generate_all(workers=12)

14098 clips, 0 already done, 14098 to compute


spectrograms:   0%|          | 0/14098 [00:00<?, ?it/s]

spectrograms:   7%|▋         | 1025/14098 [00:32<06:55, 31.43it/s]

spectrograms:   7%|▋         | 1025/14098 [00:47<06:55, 31.43it/s]

spectrograms:  19%|█▉        | 2721/14098 [01:02<04:09, 45.60it/s]

spectrograms:  19%|█▉        | 2721/14098 [01:17<04:09, 45.60it/s]

... (21 outputs left out)

spectrograms:  88%|████████▊ | 12433/14098 [07:08<00:57, 28.91it/s]

spectrograms:  95%|█████████▍| 13393/14098 [07:31<00:25, 27.91it/s]

spectrograms: 100%|██████████| 14098/14098 [07:46<00:00, 30.20it/s]

done, 0 errors


In [7]:
import torch
files = list(config.SPEC_DIR.rglob("*.norm.pt"))
t = torch.load(files[0], weights_only=True)
print(len(files), "spectrogram files; example", files[0].name, tuple(t.shape), f"mean {t.mean():.3f} std {t.std():.3f}")
assert len(files) == len(clean), "missing spectrograms"


14098 spectrogram files; example 61025001.wav.norm.pt (1, 128, 304) mean -0.000 std 1.000
